# Preparing dataframes for pseudobulking analysis - Sample type 
## Using DC anndata object 20250617_AllDC_bbknn.h5ad 
* sum by sample, so for each DC subset (3 dataframes) made a dataframe that has genes as rows and integration_id (samples) as columns
* then make a metadata dataframe that has two columns --> integration_id and cancer type

## Using code from NR 2025-01-13_Pseudobulking.ipynb


In [1]:
#load packages
import scanpy as sc
import os
import pandas as pd

In [2]:
#change current directory 
os.chdir('/scratch/user/uqlbukat/NRAtlas_analysis_2026/NR_code/Pseudobulk')
os.getcwd()

'/scratch/user/uqlbukat/NRAtlas_analysis_2026/NR_code/Pseudobulk'

In [4]:
# Read in the data 
#data = sc.read_h5ad('/scratch/user/uqlbukat/NRAtlas_analysis_2026/20250617_AllDC_bbknn.h5ad')

## Use the NR raw data

In [3]:
# Read in the data 
data = sc.read_h5ad('/scratch/user/uqlbukat/NRAtlas_analysis_2026/NRclean_Raw_Data.h5ad')

In [6]:
data

AnnData object with n_obs × n_vars = 498023 × 15074
    obs: 'nCount_RNA', 'nFeature_RNA', 'sample_type', 'cancer_type', 'patient_id', 'sample_id', 'percent.mt', 'site', 'sample_type_major', 'cancer_subtype', 'integration_id', 'ident', 'dataset_id', 'Technology', 'NR_annotations_simple', 'sample_type_major2', 'cancer_with_H', 'cancer_broadest', 'patient_treatment'
    var: 'name'

In [5]:
# filter to just primary tumours 
#data = data[data.obs["sample_type_major2"].isin(["primary tumour"])]

# Make pseudobulk dataframes: 

In [7]:
#subset out each DC
data_DC1 = data[data.obs["NR_annotations_simple"].isin(["cDC1"])]
data_DC2 = data[data.obs["NR_annotations_simple"].isin(["cDC2"])]
data_mregDC = data[data.obs["NR_annotations_simple"].isin(["mregDC"])]

In [8]:
#checking how many samples in each (this will have to match metadata dataframe below)
num_unique_integration_ids = data_mregDC.obs['integration_id'].nunique()
print(f"Number of unique integration_id values: {num_unique_integration_ids}")

Number of unique integration_id values: 536


In [9]:
data_DC1_matrix = data_DC1.X.toarray()
data_DC2_matrix = data_DC2.X.toarray()
data_mregDC_matrix = data_mregDC.X.toarray()

In [10]:
genes_DC1 = data_DC1.var_names
obs_DC1 = data_DC1.obs

genes_DC2 = data_DC2.var_names
obs_DC2 = data_DC2.obs

genes_mregDC = data_mregDC.var_names
obs_mregDC = data_mregDC.obs

In [11]:
# Create a DataFrame with cells as rows and genes as columns
data_DC1_df = pd.DataFrame(data_DC1_matrix, index=obs_DC1.index, columns=genes_DC1)
data_DC2_df = pd.DataFrame(data_DC2_matrix, index=obs_DC2.index, columns=genes_DC2)
data_mregDC_df = pd.DataFrame(data_mregDC_matrix, index=obs_mregDC.index, columns=genes_mregDC)

In [12]:
# Add integration_id
data_DC1_df['integration_id'] = obs_DC1['integration_id']
data_DC2_df['integration_id'] = obs_DC2['integration_id']
data_mregDC_df['integration_id'] = obs_mregDC['integration_id']

In [13]:
data_DC1_df.head()

,FAM87B,LINC00115,FAM41C,SAMD11,NOC2L,KLHL17,PLEKHN1,HES4,ISG15,AGRN,...,MT-ATP8,MT-ATP6,MT-CO3,MT-ND3,MT-ND4L,MT-ND4,MT-ND5,MT-ND6,MT-CYB,integration_id
GSE215120_AM1_ACCAGTAAGACTGGGT-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,0.0,...,0.0,13.0,29.0,5.0,0.0,11.0,5.0,1.0,15.0,GSE215120_Acral_MEL_AM1
GSE215120_AM1_ACGTCAACAAGGACTG-1,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,27.0,0.0,...,1.0,52.0,162.0,22.0,0.0,64.0,10.0,1.0,74.0,GSE215120_Acral_MEL_AM1
GSE215120_AM1_ATAACGCGTAGCGCTC-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,11.0,35.0,8.0,0.0,14.0,5.0,0.0,20.0,GSE215120_Acral_MEL_AM1
GSE215120_AM1_ATAGACCCATTACCTT-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,17.0,38.0,12.0,0.0,20.0,4.0,0.0,25.0,GSE215120_Acral_MEL_AM1
GSE215120_AM1_ATTGGTGAGTCTCGGC-1,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,5.0,0.0,...,0.0,23.0,50.0,13.0,1.0,29.0,6.0,1.0,29.0,GSE215120_Acral_MEL_AM1


In [14]:
# Group by integration_id and sum to pseudobulk
pseudobulk_DC1_df = data_DC1_df.groupby('integration_id').sum().transpose()
pseudobulk_DC2_df = data_DC2_df.groupby('integration_id').sum().transpose()
pseudobulk_mregDC_df = data_mregDC_df.groupby('integration_id').sum().transpose()

/scratch/temp/23523265/ipykernel_3502343/1587600822.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  pseudobulk_DC1_df = data_DC1_df.groupby('integration_id').sum().transpose()
/scratch/temp/23523265/ipykernel_3502343/1587600822.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  pseudobulk_DC2_df = data_DC2_df.groupby('integration_id').sum().transpose()
/scratch/temp/23523265/ipykernel_3502343/1587600822.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the fut

In [15]:
pseudobulk_DC1_df.head()

integration_id,GGSE223063_GLIO_1,GGSE223063_GLIO_3,GSE112271_HCC_Pt13,GSE112271_HCC_Pt14,GSE131907_Healthy_N0001,GSE131907_Healthy_N0006,GSE131907_Healthy_N0008,GSE131907_Healthy_N0009,GSE131907_Healthy_N0018,GSE131907_Healthy_N0019,...,PRJCA005422_HGSOC9_PT,PRJCA005422_HGSOC10_AS,PRJNA907381_MEL002_iLN,PRJNA907381_MEL002_uLN,PRJNA907381_MEL009_iLN,PRJNA907381_MEL014_iLN,PRJNA907381_MEL014_uLN,PRJNA907381_MEL018_iLN,PRJNA907381_MEL018_uLN,PRJNA907381_MEL022_iLN
FAM87B,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LINC00115,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,...,0.0,1.0,0.0,1.0,1.0,0.0,0.0,2.0,1.0,0.0
FAM41C,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,2.0,1.0,1.0,0.0,0.0,1.0,0.0
SAMD11,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
NOC2L,0.0,0.0,0.0,2.0,4.0,1.0,0.0,4.0,8.0,6.0,...,1.0,16.0,3.0,8.0,18.0,3.0,0.0,11.0,31.0,4.0


In [16]:
pseudobulk_DC2_df.head()

integration_id,GGSE223063_GLIO_1,GGSE223063_GLIO_3,GSE112271_HCC_Pt13,GSE112271_HCC_Pt14,GSE131907_Healthy_N0001,GSE131907_Healthy_N0006,GSE131907_Healthy_N0008,GSE131907_Healthy_N0009,GSE131907_Healthy_N0018,GSE131907_Healthy_N0019,...,PRJCA005422_HGSOC10_AS,PRJCA005422_HGSOC10_PT,PRJNA907381_MEL002_iLN,PRJNA907381_MEL002_uLN,PRJNA907381_MEL009_iLN,PRJNA907381_MEL014_iLN,PRJNA907381_MEL014_uLN,PRJNA907381_MEL018_iLN,PRJNA907381_MEL018_uLN,PRJNA907381_MEL022_iLN
FAM87B,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0
LINC00115,0.0,0.0,0.0,1.0,2.0,0.0,1.0,0.0,0.0,2.0,...,0.0,0.0,2.0,13.0,3.0,2.0,3.0,2.0,6.0,2.0
FAM41C,1.0,0.0,0.0,1.0,2.0,2.0,0.0,2.0,2.0,2.0,...,0.0,1.0,0.0,12.0,1.0,0.0,0.0,0.0,5.0,0.0
SAMD11,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,2.0,1.0,0.0,0.0,0.0,0.0
NOC2L,4.0,0.0,3.0,6.0,14.0,4.0,1.0,6.0,19.0,11.0,...,8.0,3.0,15.0,119.0,49.0,25.0,10.0,27.0,87.0,15.0


In [17]:
pseudobulk_mregDC_df.head()

integration_id,GGSE223063_GLIO_1,GSE112271_HCC_Pt13,GSE112271_HCC_Pt14,GSE131907_Healthy_N0001,GSE131907_Healthy_N0006,GSE131907_Healthy_N0008,GSE131907_Healthy_N0009,GSE131907_Healthy_N0018,GSE131907_Healthy_N0019,GSE131907_Healthy_N0020,...,PRJCA005422_HGSOC10_AS,PRJCA005422_HGSOC10_PT,PRJNA907381_MEL002_iLN,PRJNA907381_MEL002_uLN,PRJNA907381_MEL009_iLN,PRJNA907381_MEL014_iLN,PRJNA907381_MEL014_uLN,PRJNA907381_MEL018_iLN,PRJNA907381_MEL018_uLN,PRJNA907381_MEL022_iLN
FAM87B,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LINC00115,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,2.0,0.0
FAM41C,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,2.0,0.0,...,0.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0,3.0,0.0
SAMD11,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
NOC2L,2.0,0.0,3.0,5.0,1.0,2.0,0.0,6.0,3.0,2.0,...,0.0,0.0,1.0,27.0,1.0,2.0,1.0,4.0,23.0,0.0


In [18]:
os.getcwd()

'/scratch/user/uqlbukat/NRAtlas_analysis_2026/NR_code/Pseudobulk'

In [42]:
# save as csv
pseudobulk_DC1_df.to_csv("pseudobulk_DC1_data.csv", index=True)
pseudobulk_DC2_df.to_csv("pseudobulk_DC2_data.csv", index=True)
pseudobulk_mregDC_df.to_csv("pseudobulk_mregDC_data.csv", index=True)

# Make metadata dataframes

In [19]:
data_DC1

View of AnnData object with n_obs × n_vars = 6730 × 15074
    obs: 'nCount_RNA', 'nFeature_RNA', 'sample_type', 'cancer_type', 'patient_id', 'sample_id', 'percent.mt', 'site', 'sample_type_major', 'cancer_subtype', 'integration_id', 'ident', 'dataset_id', 'Technology', 'NR_annotations_simple', 'sample_type_major2', 'cancer_with_H', 'cancer_broadest', 'patient_treatment'
    var: 'name'

In [20]:
metadata_DC1 = data_DC1.obs[['integration_id', 'sample_type_major2']].groupby('integration_id').first()
metadata_DC2 = data_DC2.obs[['integration_id', 'sample_type_major2']].groupby('integration_id').first()
metadata_mregDC = data_mregDC.obs[['integration_id', 'sample_type_major2']].groupby('integration_id').first()

/scratch/temp/23523265/ipykernel_3502343/4277918564.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  metadata_DC1 = data_DC1.obs[['integration_id', 'sample_type_major2']].groupby('integration_id').first()
/scratch/temp/23523265/ipykernel_3502343/4277918564.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  metadata_DC2 = data_DC2.obs[['integration_id', 'sample_type_major2']].groupby('integration_id').first()
/scratch/temp/23523265/ipykernel_3502343/4277918564.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to r

In [21]:
metadata_mregDC.head()

,sample_type_major2
integration_id,
GGSE223063_GLIO_1,primary tumour
GSE112271_HCC_Pt13,primary tumour
GSE112271_HCC_Pt14,primary tumour
GSE131907_Healthy_N0001,healthy
GSE131907_Healthy_N0006,healthy


In [22]:
metadata_mregDC.shape

(536, 1)

In [23]:
# set the index (integration_id) to be a column
metadata_DC1 = metadata_DC1.reset_index()
metadata_DC2 = metadata_DC2.reset_index()
metadata_mregDC = metadata_mregDC.reset_index()

In [48]:
metadata_mregDC.head()

,integration_id,cancer_broadest
0,GGSE223063_GLIO_1,GBM
1,GSE112271_HCC_Pt13,HCC
2,GSE112271_HCC_Pt14,HCC
3,GSE131907_Healthy_N0001,NA
4,GSE131907_Healthy_N0006,NA


In [24]:
# save as csv
metadata_DC1.to_csv("pseudobulk_sampletype_metadata_DC1.csv", index=False)
metadata_DC2.to_csv("pseudobulk_sampletype_metadata_DC2.csv", index=False)
metadata_mregDC.to_csv("pseudobulk_sampletype_metadata_mregDC.csv", index=False)